In [ ]:
from dataclasses import dataclass, field
import json
from pathlib import Path
import time
from typing import Any, Protocol

import pandas as pd
from tqdm import tqdm
import yaml

from kibad_crawler.config import MAX_NUM_BATCH_PAPERS, RAW_DATA_DIR
from kibad_crawler.papers import PaperSearch, get_papers
from kibad_crawler.semantic_scholar import academic_graph, datasets
from kibad_crawler.utils import chunker

In [ ]:
release_id = datasets.get_latest_release()
print(release_id)

# Publication Venues

In [ ]:
links_publication_venues = datasets.download_links(release_id, "publication-venues")
time.sleep(2)
output_path = datasets.download_dataset_from_semantic_scholar(
    url=links_publication_venues["files"][0],
    output_path=RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "publication_venues",
)
df_venues = datasets.read_publication_venues(file_path=output_path)

In [ ]:
df_venues.info()

In [ ]:
path_journals = Path("/Users/darenasc/projects/dfki/kibad-crawler/data/raw/journals.csv")


def get_issn_codes(path_journals_csv: Path) -> list:
    """Return a list of ISSN code to filter journals. It reads a CSV with ISSN
    column names and filter it's content.

    Args:
        file_path (Path): CSV file

    Returns:
        list: _description_
    """
    df_journals = pd.read_csv(path_journals_csv)
    columns = [x for x in df_journals.columns if "issn" in str(x).lower()]
    issn = []
    for column in columns:
        issn += [x for x in df_journals[column].unique() if isinstance(x, str)]
    return issn


issn = get_issn_codes(path_journals)
publication_venue_ids = df_venues[df_venues["issn"].isin(issn)].id.unique()
print(issn[:5])
print(len(publication_venue_ids))

In [ ]:
search_params = PaperSearch(publication_venue_ids=publication_venue_ids)
len(search_params.publication_venue_ids)

# Papers

In [ ]:
links_papers = datasets.download_links(release_id=release_id, dataset_name="papers")

In [ ]:
output_path_papers = RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "papers"

In [ ]:
for url in links_papers["files"][:4]:
    get_papers(url=url, output_path=output_path_papers, search_params=search_params)

# Abstracts

In [ ]:
paper_metadata_files = [x for x in output_path_papers.glob("**/*.csv")]

In [ ]:
output_path_papers

In [ ]:
# paper_files = [x for x in (output_path_papers).glob("*.csv")]
# dfs = [pd.read_csv(f) for f in paper_files]
# df_all_papers = pd.concat(dfs)

In [ ]:
def get_abstracts(path: Path):
    df = pd.read_csv(path)
    temp_folder = path.name[:-7]
    paper_ids = []
    for i, r in df.iterrows():
        paper_ids.append(r["url"].split("/")[-1])

    json_path = output_path_papers / temp_folder / "json"
    json_path.mkdir(parents=True, exist_ok=True)

    for i, group in tqdm(
        enumerate(chunker(paper_ids, MAX_NUM_BATCH_PAPERS)),
        total=int(len(paper_ids) / MAX_NUM_BATCH_PAPERS),
        desc="Downloading metadata",
    ):
        response_batch = academic_graph.batch(group)

        json_file = json_path / f"papers-metadata-{i}.json"
        with open(str(json_file), "w", encoding="utf-8") as f:
            json.dump(response_batch, f)

In [ ]:
get_abstracts(paper_metadata_files[1])

In [ ]:
df = pd.read_csv(paper_metadata_files[0])
temp_folder = paper_metadata_files[0].name[:-7]
paper_ids = []
for i, r in df.iterrows():
    paper_ids.append(r["url"].split("/")[-1])

In [ ]:
len(paper_ids)

In [ ]:
json_path = output_path_papers / temp_folder / "json"
json_path.mkdir(parents=True, exist_ok=True)

In [ ]:
for i, group in tqdm(
    enumerate(chunker(paper_ids, MAX_NUM_BATCH_PAPERS)),
    total=int(len(paper_ids) / MAX_NUM_BATCH_PAPERS),
    desc="Downloading metadata",
):
    # response_batch = self.download_bulk_paper(group)
    response_batch = academic_graph.batch(group)

    json_file = json_path / f"papers-metadata-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch, f)

# Relevance Filter

In [ ]:
file_path = "/Users/darenasc/projects/dfki/kibad-crawler/notebooks/query_strings.yml"
with open(file_path, "r") as file:
    yaml_data = yaml.safe_load(file)
    # words = set()
    # for word_list in yaml_data.get("words", []):
    #     words.update(word_list)

# yaml_data

In [ ]:
yaml_data["Biodiversity"]["words"]

In [ ]:
@dataclass
class Concept(Protocol):
    name: str
    words: list[str]

    def check(self, text: str) -> bool: ...


def get_biodiversity_words() -> list[str]:
    return ["species richness", "species number", "number of species"]


@dataclass
class ConceptBiodiversity:
    name: str = "Biodiversity"
    words: list[str] = field(default_factory=get_biodiversity_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False

    def __call__(self, *args: Any, **kwds: Any) -> Any:
        return self.check(*args)


def get_deutschland_words() -> list[str]:
    return ["German", "central europ", "Lower Saxony"]


@dataclass
class ConceptDeutschland:
    name: str = "Deutschland"
    words: list[str] = field(default_factory=get_deutschland_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False


def get_lebensraum_words() -> list[str]:
    return ["Forest", "wood", "stock of trees"]


@dataclass
class ConceptLebensraum:
    name: str = "Lebensraum"
    words: list[str] = field(default_factory=get_lebensraum_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False


def get_wald_trends_words():
    return ["trend", "chang", "decreas"]


@dataclass
class ConceptWaldTrends:
    name: str = "WaldTrends"
    words: list = field(default_factory=get_wald_trends_words)

    def check(self, text: str) -> bool:
        concept_biodiversity = ConceptBiodiversity()
        concept_deutschland = ConceptDeutschland()
        concept_lebensraum = ConceptLebensraum()
        check_wald_trend = any(word in text for word in self.words)
        return (
            concept_biodiversity.check(text=text)
            and concept_deutschland.check(text=text)
            and concept_lebensraum.check(text=text)
            and check_wald_trend
        )

In [ ]:
concept_wald_trends = ConceptWaldTrends()
print(concept_wald_trends.check("richness trend"))
print(concept_wald_trends.check("species richness trend"))
print(concept_wald_trends.check("species richness trend German Forest"))

In [ ]:
# query._check("richness")
# concept_biodiversity("richness")
# concept_deutschland("richness")

In [ ]:
# @dataclass
# class Concept(Protocol):
#     name: str
#     words: list[str]

#     def _check(self, text: str) -> bool:
#         if any(word in text for word in self.words):
#             return True
#         else:
#             return False

#     def __call__(self, *args: Any, **kwds: Any) -> Any:
#         return self._check(*args)